# 🏗️ Notebook 1: Airbnb — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This lab uses **only the Python standard library + `pydantic`**. No Docker, no external services. Every cell runs on your laptop in <1s.

## 🎯 What are we building?

**Airbnb** is a two-sided marketplace for short-term rentals:

- **Hosts** list properties with photos, a price, and a calendar.
- **Guests** search by location + dates, view a listing, and **book** it.

Sounds simple. Two things make it *hard* at scale:

1. **Search** — fast "show me places near Seattle available May 1–4" across millions of listings.
2. **Booking** — two guests click *Reserve* at the same millisecond. Only one can win. No double-bookings, ever.

The rest of this lab peels back each of those problems with runnable code.

## ✅ Requirements

### Functional (the happy path)
- Host creates a **Listing** (title, photos, price, address, calendar).
- Guest searches listings by **location + check-in/check-out + #guests**.
- Guest views a listing detail page with available dates and price.
- Guest **books** a date range and pays.
- Host is notified; the calendar updates everywhere.
- Either party can **cancel** within policy.

### Non-functional (the constraints that shape the design)
| # | Requirement | Why it matters |
|---|---|---|
| 1 | **No double-bookings**, ever | Customer trust. Worth sacrificing availability briefly. |
| 2 | **Read-heavy**: ~100× more searches than bookings | Shape of cache, index, read replicas. |
| 3 | **P95 search < 300 ms** | Users bounce on slow search. |
| 4 | **Eventual consistency OK for search index** | We can tolerate a new listing showing up a few seconds late. |
| 5 | **Strong consistency for booking** | We cannot tolerate the calendar being wrong. |
| 6 | 99.9% availability | Outages = lost revenue for hosts + guests. |

> 💡 **Big idea**: search and booking have *opposite* needs. Search wants lots of cached replicas. Booking wants one authoritative writer. The architecture must give each what it wants.

## 🔢 Back-of-the-envelope (runnable)

Before drawing boxes, estimate size & load. Wrong numbers → wrong design.

In [ ]:
# Rough scale estimates — tweak the numbers to see how the system changes shape.
listings = 10_000_000
daily_active_users = 10_000_000
searches_per_user_per_day = 5
bookings_per_day = 1_000_000
avg_nights_per_booking = 3

# --- QPS ---
search_qps_avg = daily_active_users * searches_per_user_per_day / 86_400
search_qps_peak = search_qps_avg * 5           # peak factor
booking_qps_peak = bookings_per_day * 5 / 86_400

# --- Storage: the search index ---
avg_listing_bytes = 2_000                      # JSON doc incl. photos metadata
index_size_gb = listings * avg_listing_bytes / 1e9

# --- Storage: the availability table ---
# Two ways to model it. Both are billions of rows, so we shard either way --
# but they differ in what you have to *operate*.
ROW_BYTES = 40
#  (a) DENSE: pre-create a row for every listing x every day of the next year.
dense_rows = listings * 365
#  (b) SPARSE: only write a row when a night is actually taken. Most listings
#      are empty most nights, so we only pay for what is booked.
sparse_rows = bookings_per_day * avg_nights_per_booking * 365

print(f"search QPS  avg/peak : {search_qps_avg:>8.0f} / {search_qps_peak:>8.0f}")
print(f"booking QPS peak     : {booking_qps_peak:>8.1f}")
print(f"search index size    : {index_size_gb:>8.1f} GB")
print()
print(f"availability DENSE   : {dense_rows:>14,} rows  (~{dense_rows*ROW_BYTES/1e9:>4.0f} GB)")
print(f"availability SPARSE  : {sparse_rows:>14,} rows  (~{sparse_rows*ROW_BYTES/1e9:>4.0f} GB)")
print(f"implied occupancy    : {sparse_rows/dense_rows:>14.0%}   <- sparse/dense IS the occupancy rate")
print(f"sparse saves         : {1 - sparse_rows/dense_rows:>14.0%} of the rows")

# Sanity-check the assumption before trusting the answer: does 30% occupancy at
# 10M listings actually produce 1M bookings/day? If not, one of the inputs is wrong.
implied_bookings_per_day = dense_rows * (sparse_rows/dense_rows) / 365 / avg_nights_per_booking
print(f"\ncross-check: {implied_bookings_per_day:,.0f} bookings/day (input was {bookings_per_day:,})")

### What the numbers tell us

- Peak search ≈ **3k QPS** — easy for a search index (Elasticsearch/OpenSearch) across a handful of shards.
- Peak booking ≈ **~60 QPS** — a single well-tuned Postgres can handle this; the hard part is *correctness*, not throughput.
- Index ≈ **20 GB** — fits on one node. Replicate for HA, not capacity.
- Availability is **billions of rows either way**, so it gets sharded by `listing_id`
  no matter which model we pick. Sparse only saves ~70% of the rows, because that
  70% *is* the vacancy rate — a number the business would very much like to shrink.

So the choice between dense and sparse is **not** about bytes. It is about what you
have to operate:

| | Dense (row per listing per day) | Sparse (row only when sold) |
|---|---|---|
| Background jobs | needs a nightly job to extend every listing's calendar one more day, forever | none |
| "Is this night free?" | `SELECT status` — one row read | `SELECT 1 ... ` — absence of a row |
| New listing | 365 inserts | 0 inserts |
| Per-day price / host block | fits naturally in the same row | needs a second table |
| Double-booking | needs an explicit `status` check + lock | **impossible**: the primary key is the lock |

> 💡 We pick **sparse**, and the deciding vote is the last row. Notebook 2 shows how
> `PRIMARY KEY (listing_id, day)` on a sparse table turns "no double-bookings, ever"
> from application logic (which you can get wrong) into a schema constraint (which
> you cannot). We pay for it with a second `calendar_rules` table for pricing and
> host blocks.

## 🧱 High-level architecture

```
                 ┌──────────────┐
                 │  Client App  │  (web / iOS / Android)
                 └──────┬───────┘
                        │ HTTPS
                 ┌──────▼───────┐
                 │ API Gateway  │   auth, rate-limit, routing
                 └──┬────┬──┬───┘
         ┌──────────┘    │  └──────────────┐
         ▼               ▼                 ▼
  ┌────────────┐  ┌────────────┐    ┌────────────┐
  │  Search    │  │  Listing   │    │  Booking   │
  │  Service   │  │  Service   │    │  Service   │
  └─────┬──────┘  └─────┬──────┘    └─────┬──────┘
        │               │                 │
        ▼               ▼                 ▼
  ┌──────────┐    ┌──────────┐      ┌──────────┐
  │ Search   │    │ Postgres │      │ Postgres │
  │ Index    │◄───│ (CDC)    │      │ (bookings│
  │ (ES/OS)  │    │          │      │ + avail) │
  └──────────┘    └──────────┘      └────┬─────┘
                                          │ events
                                          ▼
                                    ┌──────────┐
                                    │ Payment  │   (Stripe)
                                    │ Service  │
                                    └──────────┘
```

### Why these pieces?

- **API Gateway** — one front door. Does auth, throttling, and routing so services stay thin.
- **Search Service** — read-only, reads from a denormalised search index built via **CDC** (Change Data Capture) from the Listing DB.
- **Listing Service** — source of truth for listing data. Owns its own DB.
- **Booking Service** — owns the calendar and bookings. Uses DB constraints to prevent double-booking (notebook 3).
- **Payment Service** — thin wrapper around Stripe. Bookings stay `pending` until payment confirms.

Services are **stateless**; state lives in the databases. Scale by adding more service replicas behind a load balancer.

## 🔄 Sequence: a booking

```
Guest ──▶ API GW ──▶ Booking Svc ──▶ DB (TX)
                                      │ check availability
                                      │ insert booking
                                      └─ commit
Booking Svc ──▶ Payment Svc ──▶ Stripe
Booking Svc ──▶ Notification Svc ──▶ Host email/SMS
Booking Svc ──▶ CDC ──▶ Search index (mark dates unavailable)
```

The transaction boundary is small: just *check-and-insert*. Payment runs outside the lock so we don't hold rows for 5 seconds waiting on Stripe.

## 📚 Where next?

- **Notebook 2** — Data model (Pydantic) and the public HTTP API. Bad schema vs. good schema for availability.
- **Notebook 3** — Deep dives with runnable code: double-booking prevention, geo search, caching, rate limiting.